In [ ]:
# ============================================
# DEEPFAKE DETECTION - T=32 TRAINING PIPELINE
# ============================================
# 4 ablation experiments: full, spatial, temporal, sequential
# Key difference from T=16: creates NEW split (T=32 dataset has fewer videos)
#
# Input dataset: dfdc02-preprocessed-t32
# GPU: P100 recommended (16GB)
# Time: ~8-10 hours

import subprocess, sys, os

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 3: Write training script
train_script = r'''
import os, sys, time, json, gc
os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import numpy as np
print(f'numpy version: {np.__version__}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!"}')
assert torch.cuda.is_available(), 'No GPU detected!'
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# =============================================
# Find T=32 dataset
# =============================================
print('\n=== Searching for T=32 dataset ===')
data_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'real' in dirs and 'fake' in dirs:
        real_p = os.path.join(root, 'real')
        fake_p = os.path.join(root, 'fake')
        rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
        fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
        if rc > 0 and fc > 0:
            # Verify this is T=32 data by checking frame count
            sample_dir = None
            for label in ['real', 'fake']:
                lp = os.path.join(root, label)
                for vd in os.listdir(lp):
                    vdp = os.path.join(lp, vd)
                    if os.path.isdir(vdp):
                        sample_dir = vdp
                        break
                if sample_dir:
                    break
            if sample_dir:
                frame_count = len([f for f in os.listdir(sample_dir) if f.endswith('.jpg') or f.endswith('.png')])
                print(f'Found: {root} (real={rc}, fake={fc})')
                print(f'Dataset: {root}')
                print(f'Real: {rc}, Fake: {fc}, Total: {rc + fc}')
                print(f'Sample video frames: {frame_count} (expected ~32)')
                data_path = root
                break

if data_path is None:
    print('ERROR: Dataset not found! Listing /kaggle/input/:')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            indent = '  ' * level
            print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
    sys.exit(1)

# =============================================
# Run 4 experiments
# =============================================
from config import Config
from train import train
from utils import save_metrics

EXPERIMENTS = [
    {'name': 'A1_full_model',    'model_type': 'full',       'fusion_type': 'adaptive'},
    {'name': 'A2_spatial_only',  'model_type': 'spatial',    'fusion_type': 'adaptive'},
    {'name': 'A3_temporal_only', 'model_type': 'temporal',   'fusion_type': 'adaptive'},
    {'name': 'A4_sequential',    'model_type': 'sequential', 'fusion_type': 'adaptive'},
]

SPLIT_FILENAME = 'split_seed42_t32.json'

all_results = []
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f'\n{"="*70}')
    print(f'[{i}/4] {exp["name"]} (model_type={exp["model_type"]}, T=32)')
    print(f'{"="*70}\n')

    cfg = Config()
    cfg.dataset_root = data_path
    cfg.dataset_name = 'dfdc02'
    cfg.model_type = exp['model_type']
    cfg.fusion_type = exp['fusion_type']
    cfg.num_frames = 32
    cfg.min_frames_per_video = 24
    cfg.max_epochs = 30
    cfg.batch_size = 8
    cfg.patience = 7
    cfg.device = 'auto'
    cfg.use_amp = True
    cfg.num_workers = 2
    cfg.pin_memory = True
    cfg.splits_dir = './splits'
    cfg.output_dir = './experiments'
    cfg.seed = 42
    cfg.unfreeze_last_n_blocks = 2
    cfg.split_filename = SPLIT_FILENAME

    # CRITICAL: First experiment creates NEW split for T=32 data,
    # subsequent experiments reuse it
    if i == 1:
        cfg.split_mode = 'random'   # Create new split
        cfg.save_split = True
    else:
        cfg.split_mode = 'fixed'    # Reuse split from experiment 1
        cfg.save_split = False

    t0 = time.time()
    try:
        metrics = train(cfg)
        metrics['experiment_name'] = exp['name']
        metrics['status'] = 'success'
        metrics['duration_min'] = round((time.time() - t0) / 60, 1)
        all_results.append(metrics)
        test = metrics.get('test', {})
        print(f'\n[OK] {exp["name"]} done in {metrics["duration_min"]} min')
        print(f'     AUC={test.get("auc",0):.4f}  Acc={test.get("accuracy",0):.4f}  F1={test.get("f1",0):.4f}')
    except Exception as e:
        elapsed = round((time.time() - t0) / 60, 1)
        print(f'\n[FAIL] {exp["name"]} after {elapsed} min: {e}')
        import traceback; traceback.print_exc()
        all_results.append({
            'experiment_name': exp['name'],
            'status': 'failed',
            'error': str(e),
            'duration_min': elapsed,
        })

    # Cleanup between experiments
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f'GPU Memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
          f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

    save_metrics(all_results, './experiments/all_results_t32.json')

# =============================================
# Summary
# =============================================
print(f'\n{"="*70}')
print('RESULTS SUMMARY (T=32)')
print(f'{"="*70}')
print(f'{"Experiment":<25} {"AUC":>8} {"Acc":>8} {"F1":>8} {"EER":>8} {"Epoch":>6} {"Time":>8}')
print('-' * 75)
for r in all_results:
    if r.get('status') == 'success':
        t = r.get('test', {})
        be = r.get('best_epoch', '?')
        print(f'{r["experiment_name"]:<25} {t.get("auc",0):>8.4f} {t.get("accuracy",0):>8.4f} '
              f'{t.get("f1",0):>8.4f} {t.get("eer",0):>8.4f} {be:>6} {r.get("duration_min",0):>7.1f}m')
    else:
        print(f'{r["experiment_name"]:<25} {"FAILED":>8} {r.get("error","")[:40]}')

# Package results for download
os.makedirs('/kaggle/working/experiments', exist_ok=True)
os.system('cp -r experiments/ /kaggle/working/experiments/ 2>/dev/null')
os.system('tar czf /kaggle/working/results_t32.tar.gz -C /kaggle/working/project experiments/ splits/ 2>/dev/null')
print(f'\nresults_t32.tar.gz ready for download')
'''

with open('/kaggle/working/run_training_t32.py', 'w') as f:
    f.write(train_script)

# Step 4: Run training in FRESH subprocess
print('\n=== Starting T=32 training in subprocess ===')
result = subprocess.run(
    [sys.executable, '/kaggle/working/run_training_t32.py'],
    cwd='/kaggle/working/project',
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
)
print(f'\nTraining subprocess exited with code: {result.returncode}')

# Copy results
if os.path.exists('/kaggle/working/project/experiments'):
    subprocess.run(['cp', '-r', '/kaggle/working/project/experiments/', '/kaggle/working/experiments/'])
    print('Results copied to /kaggle/working/experiments/')
if os.path.exists('/kaggle/working/results_t32.tar.gz'):
    print('results_t32.tar.gz is ready for download')